In [0]:
# Celda 1
%pip install statsmodels

In [0]:
# Celda 2
dbutils.library.restartPython()

In [0]:
# Celda 3: datos y listas de variables
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import roc_auc_score, roc_curve
import mlflow

df = spark.table("workspace.default.lc_woe").toPandas()
lista = spark.table("workspace.default.lc_model_variables").toPandas()

vars_a = [f"woe_{v}" for v in lista["variable"]]
vars_b = [f"woe_{v}" for v in lista.loc[lista["modelo_b_06"], "variable"]]

dev = df[df["sample"] == "DEV"]
oot = df[df["sample"] == "OOT"]
print(f"Modelo A: {len(vars_a)} vars | Modelo B: {len(vars_b)} vars | DEV: {len(dev):,} | OOT: {len(oot):,}")


In [0]:
# Celda 4: funciones de evaluación
def ks_auc(y, p):
    fpr, tpr, _ = roc_curve(y, p)
    return round(max(tpr - fpr) * 100, 2), round(roc_auc_score(y, p), 4)   # KS en %, AUC

def vif(X):
    Xc = sm.add_constant(X)
    return pd.Series([variance_inflation_factor(Xc.values, i) for i in range(1, Xc.shape[1])],
                     index=X.columns).round(2)

def confusion(y, p, tasa_rechazo):
    corte = np.quantile(p, 1 - tasa_rechazo)   # p = probabilidad de malo: se rechaza el % más alto
    rech = p >= corte
    tp = int((rech & (y == 1)).sum());  fp = int((rech & (y == 0)).sum())
    fn = int((~rech & (y == 1)).sum()); tn = int((~rech & (y == 0)).sum())
    return {"tasa_rechazo": tasa_rechazo,
            "malos_rechazados": tp, "buenos_rechazados": fp,
            "malos_aprobados": fn, "buenos_aprobados": tn,
            "recall_malos_pct": round(tp / (tp + fn) * 100, 2),
            "precision_rechazo_pct": round(tp / (tp + fp) * 100, 2),
            "buenos_rechazados_pct": round(fp / (fp + tn) * 100, 2),
            "bad_rate_aprobados_pct": round(fn / (fn + tn) * 100, 2)}


In [0]:
# Celda 5: entrenar, evaluar y registrar en MLflow
def entrenar(nombre, variables):
    with mlflow.start_run(run_name=nombre):
        y_dev, y_oot = dev["target"].values, oot["target"].values
        modelo = sm.Logit(y_dev, sm.add_constant(dev[variables])).fit(disp=0)

        p_dev = np.asarray(modelo.predict(sm.add_constant(dev[variables])))
        p_oot = np.asarray(modelo.predict(sm.add_constant(oot[variables])))
        ks_d, auc_d = ks_auc(y_dev, p_dev)
        ks_o, auc_o = ks_auc(y_oot, p_oot)

        coefs = pd.DataFrame({"coef": modelo.params.round(4), "p_value": modelo.pvalues.round(4)})
        coefs["vif"] = vif(dev[variables])
        conf = pd.DataFrame([confusion(y_oot, p_oot, t) for t in (0.10, 0.20, 0.30)])

        mlflow.log_params({"n_variables": len(variables), "variables": ",".join(variables)})
        mlflow.log_metrics({"ks_dev": ks_d, "auc_dev": auc_d, "ks_oot": ks_o, "auc_oot": auc_o})
        mlflow.log_table(coefs.reset_index(names="variable"), "coeficientes.json")
        mlflow.log_table(conf, "confusion_oot.json")

    resumen = {"modelo": nombre, "n_vars": len(variables),
               "ks_dev": ks_d, "auc_dev": auc_d, "ks_oot": ks_o, "auc_oot": auc_o}
    return modelo, coefs, conf, resumen

mod_a, coef_a, conf_a, res_a = entrenar("Modelo A - r0.7 - 16 vars", vars_a)
mod_b, coef_b, conf_b, res_b = entrenar("Modelo B - r0.6 - 14 vars", vars_b)

display(pd.DataFrame([res_a, res_b]))


In [0]:
print(pd.DataFrame([res_a, res_b]).to_string())
print(coef_a.to_string())
print(coef_b.to_string())
print(pd.concat([conf_a.assign(modelo="A"), conf_b.assign(modelo="B")]).to_string())


In [0]:
# Celda 6: coeficientes con p-value y VIF, y matrices de confusión en OOT
display(coef_a.reset_index(names="variable").assign(modelo="A"))
display(coef_b.reset_index(names="variable").assign(modelo="B"))
display(pd.concat([conf_a.assign(modelo="A"), conf_b.assign(modelo="B")]))


In [0]:
# Celda 8: modelo C = A sin mort_acc (no significativa) ni revol_bal (signo invertido)
vars_c = [v for v in vars_a if v not in ("woe_mort_acc", "woe_revol_bal")]
mod_c, coef_c, conf_c, res_c = entrenar("Modelo C - A sin mort_acc ni revol_bal - 14 vars", vars_c)

print(pd.DataFrame([res_a, res_b, res_c]).to_string())
print(coef_c.to_string())
print(conf_c.to_string())

#display(pd.concat([conf_a.assign(modelo="A"), conf_b.assign(modelo="B"), conf_b.assign(modelo="C")]))